# Test 1: Gradient Magnitude Distribution (Region Hypothesis)

This notebook tests the **Region Hypothesis** for feature encoding by analyzing the distribution of decoder gradient norms.

## Hypotheses

For each feature $i$ and sample embedding $h$, we compute:
$$g_i(h) = \left\| \frac{\partial \hat{x}_i}{\partial h} \right\|$$

**Predictions:**
- **Direction hypothesis**: $g_i(h)$ is approximately constant (equals $\|w_i\|$ for linear decoder)
- **Manifold hypothesis**: $g_i(h)$ is consistently nonzero everywhere, may vary smoothly
- **Region hypothesis**: $g_i(h)$ is **bimodal** - near-zero in region interiors, large at ReLU boundaries

## Architectures Compared
1. **TiedLinearRelu**: Linear encoder, ReLU on decoder output (creates genuine polytopes)
2. **TiedMLPEncoder**: MLP with LeakyReLU (multiple layers of piecewise-linear regions)

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from torch import Generator
from scipy import stats

from occhio import ToyModel
from occhio.autoencoder import TiedLinearRelu, TiedMLPEncoder
from occhio.distributions.sparse import SparseUniform
from occhio.analysis import gradient_norm_distribution, perturbation_sensitivity

## 1. Visualisable Scale (n=10, m=2-3)

Start with a small model where we can directly visualize the embedding space and gradient norms.

In [ ]:
# Small scale configuration
N_FEATURES_SMALL = 10
N_HIDDEN_SMALL = 3
N_EPOCHS_SMALL = 5000
BATCH_SIZE = 512
N_SAMPLES = 2000

# Zipf distribution: p_i = 1/(i+1)
ZIPF_PROBS_SMALL = [1 / (i + 2) ** 2 for i in range(N_FEATURES_SMALL)]

print(f"Small scale: {N_FEATURES_SMALL} features -> {N_HIDDEN_SMALL} hidden")
print(f"Compression ratio: {N_FEATURES_SMALL / N_HIDDEN_SMALL:.1f}:1")
print(
    f"Zipf p_active range: [{min(ZIPF_PROBS_SMALL):.3f}, {max(ZIPF_PROBS_SMALL):.3f}]"
)

In [ ]:
def create_zipf_distribution_small(seed=42):
    """Create SparseUniform with Zipf probabilities for small scale."""
    return SparseUniform(
        N_FEATURES_SMALL,
        ZIPF_PROBS_SMALL,
        generator=Generator().manual_seed(seed),
    )

In [ ]:
# Train TiedLinearRelu (piecewise linear - ReLU on decoder output)
print("Training TiedLinearRelu (small scale)...")
relu_model_small = ToyModel(
    distribution=create_zipf_distribution_small(),
    ae=TiedLinearRelu(n_features=N_FEATURES_SMALL, n_hidden=N_HIDDEN_SMALL),
)
relu_losses_small, _ = relu_model_small.fit(
    n_epochs=N_EPOCHS_SMALL, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {relu_losses_small[-1]:.6f}")

# Train TiedMLPEncoder (MLP with LeakyReLU)
print("\nTraining TiedMLPEncoder (small scale)...")
mlp_model_small = ToyModel(
    distribution=create_zipf_distribution_small(),
    ae=TiedMLPEncoder(dims=[N_FEATURES_SMALL, N_HIDDEN_SMALL * 4, N_HIDDEN_SMALL]),
)
mlp_losses_small, _ = mlp_model_small.fit(
    n_epochs=N_EPOCHS_SMALL, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {mlp_losses_small[-1]:.6f}")

In [ ]:
# Generate test samples and compute gradient norms
test_dist_small = create_zipf_distribution_small(seed=999)
test_samples_small = test_dist_small.sample(N_SAMPLES)

print("Computing gradient norms (small scale)...")
relu_grad_norms_small = gradient_norm_distribution(relu_model_small, test_samples_small)
mlp_grad_norms_small = gradient_norm_distribution(mlp_model_small, test_samples_small)

print(f"Gradient norms shape: {relu_grad_norms_small.shape}")
print(
    f"TiedLinearRelu - mean: {relu_grad_norms_small.mean():.4f}, std: {relu_grad_norms_small.std():.4f}"
)
print(
    f"TiedMLPEncoder - mean: {mlp_grad_norms_small.mean():.4f}, std: {mlp_grad_norms_small.std():.4f}"
)

### 1.1 Gradient Norm Histograms

Compare the distribution of gradient norms across both architectures.

In [ ]:
def compute_bimodality_coefficient(data):
    """Compute Sarle's bimodality coefficient.

    BC = (skewness^2 + 1) / kurtosis
    BC > 5/9 suggests bimodality.
    """
    data = np.array(data)
    n = len(data)
    if n < 4:
        return np.nan
    skew = stats.skew(data)
    # Excess kurtosis + 3 to get raw kurtosis
    kurt = stats.kurtosis(data, fisher=False)  # Fisher=False gives Pearson's kurtosis
    if kurt == 0:
        return np.nan
    bc = (skew**2 + 1) / kurt
    return bc


def compute_dead_gradient_fraction(grad_norms, threshold_factor=0.1):
    """Fraction of samples with gradient norm < threshold * median."""
    median_norm = np.median(grad_norms)
    threshold = threshold_factor * median_norm
    return (grad_norms < threshold).mean()


# Per-feature histogram comparison
fig = make_subplots(
    rows=2,
    cols=5,
    subplot_titles=[f"Feature {i}" for i in range(N_FEATURES_SMALL)],
    vertical_spacing=0.12,
    horizontal_spacing=0.05,
)

for feat_idx in range(N_FEATURES_SMALL):
    row = feat_idx // 5 + 1
    col = feat_idx % 5 + 1

    relu_norms = relu_grad_norms_small[:, feat_idx].detach().numpy()
    mlp_norms = mlp_grad_norms_small[:, feat_idx].detach().numpy()

    # Filter out zeros for histogram
    relu_norms_pos = relu_norms[relu_norms > 1e-8]
    mlp_norms_pos = mlp_norms[mlp_norms > 1e-8]

    fig.add_trace(
        go.Histogram(
            x=relu_norms_pos,
            name="TiedLinearRelu",
            opacity=0.6,
            marker_color="blue",
            nbinsx=30,
            showlegend=(feat_idx == 0),
        ),
        row=row,
        col=col,
    )
    fig.add_trace(
        go.Histogram(
            x=mlp_norms_pos,
            name="TiedMLPEncoder",
            opacity=0.6,
            marker_color="red",
            nbinsx=30,
            showlegend=(feat_idx == 0),
        ),
        row=row,
        col=col,
    )

fig.update_layout(
    height=500,
    title_text="Gradient Norm Distributions per Feature (Small Scale)",
    barmode="overlay",
)
fig.show()

In [ ]:
# Compute metrics for each feature
print("=" * 80)
print("GRADIENT NORM ANALYSIS - SMALL SCALE")
print("=" * 80)
print(
    f"{'Feature':<10} {'Arch':<15} {'Mean':<10} {'Std':<10} {'CV':<10} {'BC':<10} {'Dead%':<10}"
)
print("-" * 80)

for feat_idx in range(N_FEATURES_SMALL):
    for arch_name, grad_norms in [
        ("TiedLinearRelu", relu_grad_norms_small),
        ("TiedMLPEncoder", mlp_grad_norms_small),
    ]:
        norms = grad_norms[:, feat_idx].detach().numpy()
        norms_pos = norms[norms > 1e-8]

        if len(norms_pos) < 10:
            continue

        mean_norm = norms_pos.mean()
        std_norm = norms_pos.std()
        cv = std_norm / mean_norm if mean_norm > 0 else 0  # Coefficient of variation
        bc = compute_bimodality_coefficient(norms_pos)
        dead_frac = compute_dead_gradient_fraction(norms)

        print(
            f"{feat_idx:<10} {arch_name:<15} {mean_norm:<10.4f} {std_norm:<10.4f} {cv:<10.4f} {bc:<10.4f} {dead_frac * 100:<10.1f}"
        )

### 1.2 Embedding Space Heatmap (3D)

Visualize gradient norms in the 3D embedding space. Under the region hypothesis, we should see sharp boundaries.

In [ ]:
# Get embeddings for visualization using binary grid of all feature combinations
import itertools

test_samples_small = torch.tensor(
    list(itertools.product([0, 1], repeat=N_FEATURES_SMALL)), dtype=torch.float32
)

with torch.no_grad():
    relu_embeddings_small = relu_model_small.ae.encode(test_samples_small)
    mlp_embeddings_small = mlp_model_small.ae.encode(test_samples_small)

# Recompute gradient norms for the new grid samples
relu_grad_norms_small = gradient_norm_distribution(relu_model_small, test_samples_small)
mlp_grad_norms_small = gradient_norm_distribution(mlp_model_small, test_samples_small)

print(f"Grid samples: {test_samples_small.shape[0]} combinations")
print(f"Gradient norms shape: {relu_grad_norms_small.shape}")

# Pick a feature to visualize (feature 0 has highest activation probability)
FEAT_TO_VIS = 0

fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=[
        f"TiedLinearRelu - Feature {FEAT_TO_VIS}",
        f"TiedMLPEncoder - Feature {FEAT_TO_VIS}",
    ],
)

# TiedLinearRelu
relu_norms_feat = relu_grad_norms_small[:, FEAT_TO_VIS].detach().numpy()
relu_hover_text = [
    f"h: ({relu_embeddings_small[i, 0]:.3f}, {relu_embeddings_small[i, 1]:.3f}, {relu_embeddings_small[i, 2]:.3f})<br>"
    f"||∂x̂/∂h||: {relu_norms_feat[i]:.4f}<br>"
    f"x[{FEAT_TO_VIS}]: {test_samples_small[i, FEAT_TO_VIS]:.3f}"
    for i in range(len(relu_norms_feat))
]
fig.add_trace(
    go.Scatter3d(
        x=relu_embeddings_small[:, 0].numpy(),
        y=relu_embeddings_small[:, 1].numpy(),
        z=relu_embeddings_small[:, 2].numpy(),
        mode="markers",
        marker=dict(
            size=3,
            color=relu_norms_feat,
            colorscale="Viridis",
            colorbar=dict(title="||∂x̂/∂h||", x=0.45),
            opacity=0.6,
        ),
        name="TiedLinearRelu",
        hovertext=relu_hover_text,
        hoverinfo="text",
    ),
    row=1,
    col=1,
)

# TiedMLPEncoder
mlp_norms_feat = mlp_grad_norms_small[:, FEAT_TO_VIS].detach().numpy()
mlp_hover_text = [
    f"h: ({mlp_embeddings_small[i, 0]:.3f}, {mlp_embeddings_small[i, 1]:.3f}, {mlp_embeddings_small[i, 2]:.3f})<br>"
    f"||∂x̂/∂h||: {mlp_norms_feat[i]:.4f}<br>"
    f"x[{FEAT_TO_VIS}]: {test_samples_small[i, FEAT_TO_VIS]:.3f}"
    for i in range(len(mlp_norms_feat))
]
fig.add_trace(
    go.Scatter3d(
        x=mlp_embeddings_small[:, 0].numpy(),
        y=mlp_embeddings_small[:, 1].numpy(),
        z=mlp_embeddings_small[:, 2].numpy(),
        mode="markers",
        marker=dict(
            size=3,
            color=mlp_norms_feat,
            colorscale="Viridis",
            colorbar=dict(title="||∂x̂/∂h||", x=1.0),
            opacity=0.6,
        ),
        name="TiedMLPEncoder",
        hovertext=mlp_hover_text,
        hoverinfo="text",
    ),
    row=1,
    col=2,
)

fig.update_layout(
    height=500,
    title_text=f"Gradient Norm in Embedding Space (Feature {FEAT_TO_VIS})",
)
fig.show()

In [ ]:
# 3D scatter: color = most active feature, opacity = gradient magnitude
import colorsys


def hsl_to_rgba(hue_deg, s, l, alpha):
    """Convert HSL + alpha to an rgba() CSS string."""
    r, g, b = colorsys.hls_to_rgb(hue_deg / 360, l, s)
    return f"rgba({int(r * 255)},{int(g * 255)},{int(b * 255)},{alpha:.3f})"


fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=[
        "TiedLinearRelu – Most Active Feature",
        "TiedMLPEncoder – Most Active Feature",
    ],
)

for col_idx, (embeddings, grad_norms_t, name) in enumerate(
    [
        (relu_embeddings_small, relu_grad_norms_small, "TiedLinearRelu"),
        (mlp_embeddings_small, mlp_grad_norms_small, "TiedMLPEncoder"),
    ],
    start=1,
):
    norms_np = grad_norms_t.detach().numpy()
    most_active = norms_np.argmax(axis=1)
    magnitudes = norms_np.max(axis=1)
    norm_mag = magnitudes / (magnitudes.max() + 1e-8)  # [0, 1], avoid div by zero
    emb = embeddings.detach().numpy()

    # Always add all features to ensure complete legend
    for feat in range(N_FEATURES_SMALL):
        mask = most_active == feat
        hue = feat * 360 / N_FEATURES_SMALL

        if mask.sum() == 0:
            # Add invisible placeholder trace for legend
            fig.add_trace(
                go.Scatter3d(
                    x=[None],
                    y=[None],
                    z=[None],
                    mode="markers",
                    marker=dict(size=3, color=hsl_to_rgba(hue, 0.7, 0.5, 1.0)),
                    name=f"F{feat}",
                    legendgroup=f"F{feat}",
                    showlegend=(col_idx == 1),
                ),
                row=1,
                col=col_idx,
            )
        else:
            idxs = np.where(mask)[0]
            point_colors = [
                hsl_to_rgba(hue, 0.7, 0.5, float(norm_mag[i])) for i in idxs
            ]

            fig.add_trace(
                go.Scatter3d(
                    x=emb[mask, 0],
                    y=emb[mask, 1],
                    z=emb[mask, 2],
                    mode="markers",
                    marker=dict(size=3, color=point_colors),
                    name=f"F{feat}",
                    legendgroup=f"F{feat}",
                    showlegend=(col_idx == 1),
                ),
                row=1,
                col=col_idx,
            )

fig.update_layout(
    height=550,
    title_text="Most Active Feature in Embedding Space (opacity = magnitude)",
)
fig.show()

In [ ]:
# Plot feature embeddings as arrows (W matrix columns) for both architectures
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=[
        "TiedLinearRelu - Feature Embeddings",
        "TiedMLPEncoder - Feature Embeddings",
    ],
)

# Get W matrices (decoder weights)
relu_W = relu_model_small.W.detach().numpy()  # [n_hidden, n_features]
mlp_W = mlp_model_small.W.detach().numpy()

# Color scale
colors = [
    f"hsl({int(i * 360 / N_FEATURES_SMALL)}, 70%, 50%)" for i in range(N_FEATURES_SMALL)
]

# Add arrows for TiedLinearRelu
for i in range(N_FEATURES_SMALL):
    fig.add_trace(
        go.Cone(
            x=[relu_W[0, i]],
            y=[relu_W[1, i]],
            z=[relu_W[2, i]],
            u=[relu_W[0, i]],
            v=[relu_W[1, i]],
            w=[relu_W[2, i]],
            sizemode="absolute",
            sizeref=0.15,
            showscale=False,
            colorscale=[[0, colors[i]], [1, colors[i]]],
            hovertext=f"Feature {i}<br>w: ({relu_W[0, i]:.3f}, {relu_W[1, i]:.3f}, {relu_W[2, i]:.3f})<br>||w||: {np.linalg.norm(relu_W[:, i]):.3f}",
            hoverinfo="text",
            name=f"F{i}",
        ),
        row=1,
        col=1,
    )
    # Line from origin to tip
    fig.add_trace(
        go.Scatter3d(
            x=[0, relu_W[0, i]],
            y=[0, relu_W[1, i]],
            z=[0, relu_W[2, i]],
            mode="lines",
            line=dict(color=colors[i], width=4),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )

# Add arrows for TiedMLPEncoder
for i in range(N_FEATURES_SMALL):
    fig.add_trace(
        go.Cone(
            x=[mlp_W[0, i]],
            y=[mlp_W[1, i]],
            z=[mlp_W[2, i]],
            u=[mlp_W[0, i]],
            v=[mlp_W[1, i]],
            w=[mlp_W[2, i]],
            sizemode="absolute",
            sizeref=0.15,
            showscale=False,
            colorscale=[[0, colors[i]], [1, colors[i]]],
            hovertext=f"Feature {i}<br>w: ({mlp_W[0, i]:.3f}, {mlp_W[1, i]:.3f}, {mlp_W[2, i]:.3f})<br>||w||: {np.linalg.norm(mlp_W[:, i]):.3f}",
            hoverinfo="text",
            name=f"F{i}",
        ),
        row=1,
        col=2,
    )
    # Line from origin to tip
    fig.add_trace(
        go.Scatter3d(
            x=[0, mlp_W[0, i]],
            y=[0, mlp_W[1, i]],
            z=[0, mlp_W[2, i]],
            mode="lines",
            line=dict(color=colors[i], width=4),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1,
        col=2,
    )

# Add origin markers
for col in [1, 2]:
    fig.add_trace(
        go.Scatter3d(
            x=[0],
            y=[0],
            z=[0],
            mode="markers",
            marker=dict(size=5, color="black"),
            showlegend=False,
        ),
        row=1,
        col=col,
    )

fig.update_layout(
    height=500,
    title_text="Feature Embedding Vectors (W columns)",
    showlegend=False,
)
fig.show()

### 1.3 Gradient Norm vs Feature Activation

Under direction/manifold hypothesis, gradient norm shouldn't depend on feature value.
Under region hypothesis, we might see structure.

In [ ]:
fig = make_subplots(
    rows=2,
    cols=5,
    subplot_titles=[f"Feature {i}" for i in range(N_FEATURES_SMALL)],
    vertical_spacing=0.12,
    horizontal_spacing=0.05,
)

for feat_idx in range(N_FEATURES_SMALL):
    row = feat_idx // 5 + 1
    col = feat_idx % 5 + 1

    feat_values = test_samples_small[:, feat_idx].numpy()
    mlp_norms = mlp_grad_norms_small[:, feat_idx].detach().numpy()

    fig.add_trace(
        go.Scatter(
            x=feat_values,
            y=mlp_norms,
            mode="markers",
            marker=dict(size=3, opacity=0.3),
            showlegend=False,
        ),
        row=row,
        col=col,
    )

fig.update_xaxes(title_text="Feature Value", row=2)
fig.update_yaxes(title_text="Gradient Norm", col=1)
fig.update_layout(
    height=500,
    title_text="Gradient Norm vs Feature Activation (TiedMLPEncoder, Small Scale)",
)
fig.show()

In [ ]:
fig = make_subplots(
    rows=2,
    cols=5,
    subplot_titles=[f"Feature {i}" for i in range(N_FEATURES_SMALL)],
    vertical_spacing=0.12,
    horizontal_spacing=0.05,
)

for feat_idx in range(N_FEATURES_SMALL):
    row = feat_idx // 5 + 1
    col = feat_idx % 5 + 1

    feat_values = test_samples_small[:, feat_idx].numpy()
    relu_norms = relu_grad_norms_small[:, feat_idx].detach().numpy()

    fig.add_trace(
        go.Scatter(
            x=feat_values,
            y=relu_norms,
            mode="markers",
            marker=dict(size=3, opacity=0.3),
            showlegend=False,
        ),
        row=row,
        col=col,
    )

fig.update_xaxes(title_text="Feature Value", row=2)
fig.update_yaxes(title_text="Gradient Norm", col=1)
fig.update_layout(
    height=500,
    title_text="Gradient Norm vs Feature Activation (TiedLinearRelu, Small Scale)",
)
fig.show()

## 2. Medium Scale (n=200, m=20)

Repeat the analysis at larger scale for statistical power.

In [ ]:
# Medium scale configuration
N_FEATURES_MED = 200
N_HIDDEN_MED = 5
N_EPOCHS_MED = 25000
N_SAMPLES_MED = 5000

ZIPF_PROBS_MED = [1 / (i + 1) for i in range(N_FEATURES_MED)]

print(f"Medium scale: {N_FEATURES_MED} features -> {N_HIDDEN_MED} hidden")
print(f"Compression ratio: {N_FEATURES_MED / N_HIDDEN_MED:.1f}:1")


def create_zipf_distribution_medium(seed=42):
    return SparseUniform(
        N_FEATURES_MED,
        ZIPF_PROBS_MED,
        generator=Generator().manual_seed(seed),
    )

In [ ]:
# Train TiedLinearRelu (medium scale)
print("Training TiedLinearRelu (medium scale)...")
relu_model_med = ToyModel(
    distribution=create_zipf_distribution_medium(),
    ae=TiedLinearRelu(n_features=N_FEATURES_MED, n_hidden=N_HIDDEN_MED),
)
relu_losses_med, _ = relu_model_med.fit(
    n_epochs=N_EPOCHS_MED, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {relu_losses_med[-1]:.6f}")

# Train TiedMLPEncoder (medium scale)
print("\nTraining TiedMLPEncoder (medium scale)...")
mlp_model_med = ToyModel(
    distribution=create_zipf_distribution_medium(),
    ae=TiedMLPEncoder(dims=[N_FEATURES_MED, N_HIDDEN_MED * 4, N_HIDDEN_MED]),
)
mlp_losses_med, _ = mlp_model_med.fit(
    n_epochs=N_EPOCHS_MED, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {mlp_losses_med[-1]:.6f}")

In [ ]:
# Plot training losses
fig = go.Figure()
fig.add_trace(go.Scatter(y=relu_losses_med, name="TiedLinearRelu", mode="lines"))
fig.add_trace(go.Scatter(y=mlp_losses_med, name="TiedMLPEncoder", mode="lines"))
fig.update_layout(
    title="Training Loss (Medium Scale)",
    xaxis_title="Epoch",
    yaxis_title="Loss",
    yaxis_type="log",
    height=400,
)
fig.show()

In [ ]:
# Generate test samples and compute gradient norms
test_dist_med = create_zipf_distribution_medium(seed=999)
test_samples_med = test_dist_med.sample(N_SAMPLES_MED)

print("Computing gradient norms (medium scale)...")
relu_grad_norms_med = gradient_norm_distribution(relu_model_med, test_samples_med)
print("TiedLinearRelu done.")
mlp_grad_norms_med = gradient_norm_distribution(mlp_model_med, test_samples_med)
print("TiedMLPEncoder done.")

print(f"\nGradient norms shape: {relu_grad_norms_med.shape}")
print(
    f"TiedLinearRelu - mean: {relu_grad_norms_med.mean():.4f}, std: {relu_grad_norms_med.std():.4f}"
)
print(
    f"TiedMLPEncoder - mean: {mlp_grad_norms_med.mean():.4f}, std: {mlp_grad_norms_med.std():.4f}"
)

### 2.1 Per-Feature Metrics (Medium Scale)

In [ ]:
# Compute per-feature metrics
def compute_feature_metrics(grad_norms, n_features):
    """Compute metrics for each feature."""
    metrics = {
        "mean": [],
        "std": [],
        "cv": [],
        "bimodality": [],
        "dead_frac": [],
    }

    for feat_idx in range(n_features):
        norms = grad_norms[:, feat_idx].detach().numpy()
        norms_pos = norms[norms > 1e-8]

        if len(norms_pos) < 10:
            for k in metrics:
                metrics[k].append(np.nan)
            continue

        mean_norm = norms_pos.mean()
        std_norm = norms_pos.std()

        metrics["mean"].append(mean_norm)
        metrics["std"].append(std_norm)
        metrics["cv"].append(std_norm / mean_norm if mean_norm > 0 else 0)
        metrics["bimodality"].append(compute_bimodality_coefficient(norms_pos))
        metrics["dead_frac"].append(compute_dead_gradient_fraction(norms))

    return {k: np.array(v) for k, v in metrics.items()}


relu_metrics_med = compute_feature_metrics(relu_grad_norms_med, N_FEATURES_MED)
mlp_metrics_med = compute_feature_metrics(mlp_grad_norms_med, N_FEATURES_MED)

print("Per-feature metrics computed.")

In [ ]:
# Compare metrics across architectures
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "Coefficient of Variation (CV)",
        "Bimodality Coefficient",
        "Dead Gradient Fraction",
        "Mean Gradient Norm",
    ],
)

features = list(range(N_FEATURES_MED))

# CV comparison
fig.add_trace(
    go.Scatter(
        x=features,
        y=relu_metrics_med["cv"],
        name="TiedLinearRelu",
        mode="lines",
        line=dict(color="blue"),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=features,
        y=mlp_metrics_med["cv"],
        name="TiedMLPEncoder",
        mode="lines",
        line=dict(color="red"),
    ),
    row=1,
    col=1,
)

# Bimodality
fig.add_trace(
    go.Scatter(
        x=features,
        y=relu_metrics_med["bimodality"],
        name="TiedLinearRelu",
        mode="lines",
        line=dict(color="blue"),
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=features,
        y=mlp_metrics_med["bimodality"],
        name="TiedMLPEncoder",
        mode="lines",
        line=dict(color="red"),
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_hline(
    y=5 / 9, line_dash="dash", annotation_text="BC=5/9 threshold", row=1, col=2
)

# Dead fraction
fig.add_trace(
    go.Scatter(
        x=features,
        y=relu_metrics_med["dead_frac"],
        name="TiedLinearRelu",
        mode="lines",
        line=dict(color="blue"),
        showlegend=False,
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=features,
        y=mlp_metrics_med["dead_frac"],
        name="TiedMLPEncoder",
        mode="lines",
        line=dict(color="red"),
        showlegend=False,
    ),
    row=2,
    col=1,
)

# Mean gradient norm
fig.add_trace(
    go.Scatter(
        x=features,
        y=relu_metrics_med["mean"],
        name="TiedLinearRelu",
        mode="lines",
        line=dict(color="blue"),
        showlegend=False,
    ),
    row=2,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=features,
        y=mlp_metrics_med["mean"],
        name="TiedMLPEncoder",
        mode="lines",
        line=dict(color="red"),
        showlegend=False,
    ),
    row=2,
    col=2,
)

fig.update_xaxes(title_text="Feature Index", row=2)
fig.update_layout(
    height=600, title_text="Gradient Norm Metrics by Feature (Medium Scale)"
)
fig.show()

In [ ]:
# Box plot comparison of key metrics
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "CV Distribution",
        "Bimodality Coefficient",
        "Dead Gradient Fraction",
    ],
)

# CV
fig.add_trace(
    go.Box(
        y=relu_metrics_med["cv"][~np.isnan(relu_metrics_med["cv"])],
        name="TiedLinearRelu",
        marker_color="blue",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Box(
        y=mlp_metrics_med["cv"][~np.isnan(mlp_metrics_med["cv"])],
        name="TiedMLPEncoder",
        marker_color="red",
    ),
    row=1,
    col=1,
)

# Bimodality
fig.add_trace(
    go.Box(
        y=relu_metrics_med["bimodality"][~np.isnan(relu_metrics_med["bimodality"])],
        name="TiedLinearRelu",
        marker_color="blue",
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Box(
        y=mlp_metrics_med["bimodality"][~np.isnan(mlp_metrics_med["bimodality"])],
        name="TiedMLPEncoder",
        marker_color="red",
        showlegend=False,
    ),
    row=1,
    col=2,
)

# Dead fraction
fig.add_trace(
    go.Box(
        y=relu_metrics_med["dead_frac"][~np.isnan(relu_metrics_med["dead_frac"])],
        name="TiedLinearRelu",
        marker_color="blue",
        showlegend=False,
    ),
    row=1,
    col=3,
)
fig.add_trace(
    go.Box(
        y=mlp_metrics_med["dead_frac"][~np.isnan(mlp_metrics_med["dead_frac"])],
        name="TiedMLPEncoder",
        marker_color="red",
        showlegend=False,
    ),
    row=1,
    col=3,
)

fig.update_layout(
    height=400, title_text="Distribution of Gradient Norm Metrics (Medium Scale)"
)
fig.show()

### 2.2 Metrics vs Feature Probability

Do region-like properties correlate with feature frequency?

In [ ]:
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "CV vs p_active",
        "Bimodality vs p_active",
        "Dead Fraction vs p_active",
    ],
)

# CV vs probability
fig.add_trace(
    go.Scatter(
        x=ZIPF_PROBS_MED,
        y=relu_metrics_med["cv"],
        mode="markers",
        name="TiedLinearRelu",
        marker=dict(size=5, opacity=0.6, color="blue"),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=ZIPF_PROBS_MED,
        y=mlp_metrics_med["cv"],
        mode="markers",
        name="TiedMLPEncoder",
        marker=dict(size=5, opacity=0.6, color="red"),
    ),
    row=1,
    col=1,
)

# Bimodality vs probability
fig.add_trace(
    go.Scatter(
        x=ZIPF_PROBS_MED,
        y=relu_metrics_med["bimodality"],
        mode="markers",
        name="TiedLinearRelu",
        marker=dict(size=5, opacity=0.6, color="blue"),
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=ZIPF_PROBS_MED,
        y=mlp_metrics_med["bimodality"],
        mode="markers",
        name="TiedMLPEncoder",
        marker=dict(size=5, opacity=0.6, color="red"),
        showlegend=False,
    ),
    row=1,
    col=2,
)

# Dead fraction vs probability
fig.add_trace(
    go.Scatter(
        x=ZIPF_PROBS_MED,
        y=relu_metrics_med["dead_frac"],
        mode="markers",
        name="TiedLinearRelu",
        marker=dict(size=5, opacity=0.6, color="blue"),
        showlegend=False,
    ),
    row=1,
    col=3,
)
fig.add_trace(
    go.Scatter(
        x=ZIPF_PROBS_MED,
        y=mlp_metrics_med["dead_frac"],
        mode="markers",
        name="TiedMLPEncoder",
        marker=dict(size=5, opacity=0.6, color="red"),
        showlegend=False,
    ),
    row=1,
    col=3,
)

fig.update_xaxes(type="log", title_text="p_active", row=1)
fig.update_layout(height=400, title_text="Gradient Norm Metrics vs Feature Probability")
fig.show()

### 2.3 Aggregate Gradient Norm Histograms

Pool gradient norms across all features and samples.

In [ ]:
# Flatten all gradient norms
relu_all_norms = relu_grad_norms_med.flatten().detach().numpy()
mlp_all_norms = mlp_grad_norms_med.flatten().detach().numpy()

# Filter out exact zeros
relu_all_pos = relu_all_norms[relu_all_norms > 1e-8]
mlp_all_pos = mlp_all_norms[mlp_all_norms > 1e-8]

fig = make_subplots(rows=1, cols=2, subplot_titles=["TiedLinearRelu", "TiedMLPEncoder"])

fig.add_trace(
    go.Histogram(
        x=relu_all_pos, nbinsx=100, name="TiedLinearRelu", marker_color="blue"
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Histogram(x=mlp_all_pos, nbinsx=100, name="TiedMLPEncoder", marker_color="red"),
    row=1,
    col=2,
)

fig.update_xaxes(title_text="Gradient Norm ||∂x̂/∂h||")
fig.update_yaxes(title_text="Count", col=1)
fig.update_layout(
    height=400,
    title_text="Aggregate Gradient Norm Distributions (All Features, Medium Scale)",
    showlegend=False,
)
fig.show()

# Statistics
print(
    f"TiedLinearRelu: mean={relu_all_pos.mean():.4f}, std={relu_all_pos.std():.4f}, BC={compute_bimodality_coefficient(relu_all_pos):.4f}"
)
print(
    f"TiedMLPEncoder: mean={mlp_all_pos.mean():.4f}, std={mlp_all_pos.std():.4f}, BC={compute_bimodality_coefficient(mlp_all_pos):.4f}"
)

In [ ]:
# Sample 10 features across the frequency range for visualization
FEATURES_TO_PLOT = [
    0,
    2,
    3,
    4,
    5,
    20,
    40,
    60,
    61,
    62,
    63,
    64,
    65,
    80,
    100,
    120,
    140,
    160,
    180,
]

fig = make_subplots(
    rows=4,
    cols=5,
    subplot_titles=[f"Feature {i}" for i in FEATURES_TO_PLOT],
    vertical_spacing=0.12,
    horizontal_spacing=0.05,
)

for plot_idx, feat_idx in enumerate(FEATURES_TO_PLOT):
    row = plot_idx // 5 + 1
    col = plot_idx % 5 + 1

    mlp_norms = mlp_grad_norms_med[:, feat_idx].detach().numpy()
    feat_activations = test_samples_med[:, feat_idx].detach().numpy()

    # Filter out zeros
    mask = mlp_norms > 1e-8
    mlp_norms_pos = mlp_norms[mask]
    feat_activations_pos = feat_activations[mask]

    fig.add_trace(
        go.Scatter(
            x=feat_activations_pos,
            y=mlp_norms_pos,
            mode="markers",
            marker=dict(size=3, opacity=0.4),
            showlegend=False,
        ),
        row=row,
        col=col,
    )

fig.update_xaxes(title_text="Feature Activation", row=4)
fig.update_yaxes(title_text="Gradient Norm", col=1)
fig.update_layout(
    height=1000,
    title_text="Gradient Norm vs Feature Activation (TiedMLPEncoder, Medium Scale)",
)
fig.show()

## 3. Summary and Interpretation

In [ ]:
print("=" * 80)
print("TEST 1: GRADIENT MAGNITUDE DISTRIBUTION - SUMMARY")
print("=" * 80)
print()
print("Key metrics for region hypothesis detection:")
print()

# Aggregate statistics
for arch_name, metrics in [
    ("TiedLinearRelu", relu_metrics_med),
    ("TiedMLPEncoder", mlp_metrics_med),
]:
    valid_cv = metrics["cv"][~np.isnan(metrics["cv"])]
    valid_bc = metrics["bimodality"][~np.isnan(metrics["bimodality"])]
    valid_dead = metrics["dead_frac"][~np.isnan(metrics["dead_frac"])]

    print(f"{arch_name}:")
    print(
        f"  Coefficient of Variation (CV):  mean={valid_cv.mean():.4f}, std={valid_cv.std():.4f}"
    )
    print(
        f"  Bimodality Coefficient (BC):    mean={valid_bc.mean():.4f}, std={valid_bc.std():.4f}"
    )
    print(
        f"  Features with BC > 5/9:         {(valid_bc > 5 / 9).sum()} / {len(valid_bc)}"
    )
    print(
        f"  Dead Gradient Fraction:         mean={valid_dead.mean():.4f}, std={valid_dead.std():.4f}"
    )
    print()

print("Interpretation guide:")
print(
    "  - High CV (>0.5) suggests variable gradient norms (potential region structure)"
)
print("  - BC > 5/9 suggests bimodality (strong region hypothesis indicator)")
print("  - High dead gradient fraction suggests dead zones in embedding space")
print()
print("Conclusion:")

# Automated conclusion
relu_bc_frac = (
    relu_metrics_med["bimodality"][~np.isnan(relu_metrics_med["bimodality"])] > 5 / 9
).mean()
mlp_bc_frac = (
    mlp_metrics_med["bimodality"][~np.isnan(mlp_metrics_med["bimodality"])] > 5 / 9
).mean()

if mlp_bc_frac > 0.3 or relu_bc_frac > 0.3:
    print("  => SIGNIFICANT bimodality detected! Region hypothesis is plausible.")
    print("  => Proceed with Test 2 (Perturbation Sensitivity) for confirmation.")
elif mlp_bc_frac > 0.1 or relu_bc_frac > 0.1:
    print(
        "  => MODERATE bimodality detected. Region effects may be present for some features."
    )
    print("  => Consider proceeding with Test 2 selectively for high-BC features.")
else:
    print("  => LOW bimodality. Gradient norms are relatively uniform.")
    print(
        "  => Region hypothesis is unlikely. Direction or manifold hypothesis more plausible."
    )

## 4. Test 2: Perturbation Sensitivity Structure

For each sample where a feature is active, apply small random perturbations to the embedding and measure which directions cause the feature's reconstruction to change most.

**Predictions:**
- **Direction hypothesis**: Sensitive directions align with the feature's weight vector $\hat{w}_i$
- **Region hypothesis**: Sensitive directions align with ReLU boundary normals (may not align with $\hat{w}_i$)

### 4.1 Perturbation Sensitivity Analysis (Small Scale)

In [ ]:
# Compute perturbation sensitivity for all features (small scale)
N_DIRECTIONS = 200  # Sample many directions for good coverage in 3D


def compute_all_perturbation_sensitivities(
    model, test_samples, n_features, n_directions=200, step_size=0.1
):
    """Compute perturbation sensitivity for all features."""
    results = {}

    with torch.no_grad():
        embeddings = model.ae.encode(test_samples)

    for feat_idx in range(n_features):
        # Filter for samples where this feature is active
        active_mask = test_samples[:, feat_idx] > 0.1
        n_active = active_mask.sum().item()

        if n_active < 20:
            results[feat_idx] = None
            continue

        active_embeddings = embeddings[active_mask][:500]  # Limit samples

        result = perturbation_sensitivity(
            model,
            feat_idx,
            active_embeddings,
            n_directions=n_directions,
            step_size=step_size,
        )
        results[feat_idx] = result

        if feat_idx % 5 == 0:
            print(
                f"Feature {feat_idx}: alignment={result['mean_feature_alignment']:.4f}, n_samples={len(active_embeddings)}"
            )

    return results


print("Computing perturbation sensitivity (small scale)...")
print("\nTiedLinearRelu:")
relu_perturb_small = compute_all_perturbation_sensitivities(
    relu_model_small, test_samples_small, N_FEATURES_SMALL, n_directions=N_DIRECTIONS
)

print("\nTiedMLPEncoder:")
mlp_perturb_small = compute_all_perturbation_sensitivities(
    mlp_model_small, test_samples_small, N_FEATURES_SMALL, n_directions=N_DIRECTIONS
)

In [ ]:
# Compare feature direction alignment across architectures
def extract_alignment_stats(perturb_results):
    """Extract alignment statistics from perturbation results."""
    alignments = []
    max_sensitivities = []

    for feat_idx, result in perturb_results.items():
        if result is None:
            continue
        alignments.extend(result["feature_alignment"].abs().tolist())
        max_sensitivities.extend(result["max_sensitivities"].tolist())

    return np.array(alignments), np.array(max_sensitivities)


relu_align_small, relu_maxsens_small = extract_alignment_stats(relu_perturb_small)
mlp_align_small, mlp_maxsens_small = extract_alignment_stats(mlp_perturb_small)

# Alignment histogram
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Feature Direction Alignment (|cos θ|)",
        "Max Sensitivity Distribution",
    ],
)

fig.add_trace(
    go.Histogram(
        x=relu_align_small,
        nbinsx=50,
        name="TiedLinearRelu",
        marker_color="blue",
        opacity=0.6,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Histogram(
        x=mlp_align_small,
        nbinsx=50,
        name="TiedMLPEncoder",
        marker_color="red",
        opacity=0.6,
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Histogram(
        x=relu_maxsens_small,
        nbinsx=50,
        name="TiedLinearRelu",
        marker_color="blue",
        opacity=0.6,
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Histogram(
        x=mlp_maxsens_small,
        nbinsx=50,
        name="TiedMLPEncoder",
        marker_color="red",
        opacity=0.6,
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.update_xaxes(
    title_text="|cos(θ)| between max-sens dir and feature dir", row=1, col=1
)
fig.update_xaxes(title_text="Max Sensitivity (Δx̂)", row=1, col=2)
fig.update_layout(
    height=400,
    title_text="Perturbation Sensitivity Analysis (Small Scale)",
    barmode="overlay",
)
fig.show()

print(
    f"TiedLinearRelu: mean alignment={relu_align_small.mean():.4f}, std={relu_align_small.std():.4f}"
)
print(
    f"TiedMLPEncoder: mean alignment={mlp_align_small.mean():.4f}, std={mlp_align_small.std():.4f}"
)

### 4.2 Sensitivity Rose Plot (3D Visualization)

For a single feature at a single sample, visualize sensitivity as a function of perturbation direction.

In [ ]:
# 3D sensitivity visualization for a single feature
FEAT_TO_VIS_PERTURB = 0

# Get a sample for visualization
with torch.no_grad():
    mlp_emb_small = mlp_model_small.ae.encode(test_samples_small)

active_mask = test_samples_small[:, FEAT_TO_VIS_PERTURB] > 0.1
active_embeddings = mlp_emb_small[active_mask][:1]  # Single sample

if len(active_embeddings) > 0:
    # Dense sampling on the sphere for 3D
    n_theta, n_phi = 30, 60
    theta = np.linspace(0, np.pi, n_theta)
    phi = np.linspace(0, 2 * np.pi, n_phi)

    # Create grid of directions
    directions_list = []
    for t in theta:
        for p in phi:
            x = np.sin(t) * np.cos(p)
            y = np.sin(t) * np.sin(p)
            z = np.cos(t)
            directions_list.append([x, y, z])

    directions = torch.tensor(directions_list, dtype=torch.float32)

    # Compute sensitivity in each direction
    step_size = 0.1
    h = active_embeddings[0]  # [n_hidden]

    with torch.no_grad():
        baseline = mlp_model_small.ae.decode(h.unsqueeze(0))[0, FEAT_TO_VIS_PERTURB]

        sensitivities = []
        for d in directions:
            perturbed = h + step_size * d
            perturbed_recon = mlp_model_small.ae.decode(perturbed.unsqueeze(0))[
                0, FEAT_TO_VIS_PERTURB
            ]
            sensitivities.append(abs(perturbed_recon - baseline).item())

    sensitivities = np.array(sensitivities)
    directions_np = directions.numpy()

    # Scale directions by sensitivity for rose plot
    scaled_dirs = directions_np * sensitivities[:, None] * 5  # Scale for visibility

    # Get feature direction for comparison
    if mlp_perturb_small[FEAT_TO_VIS_PERTURB] is not None:
        feat_dir = (
            mlp_perturb_small[FEAT_TO_VIS_PERTURB]["feature_direction"].detach().numpy()
        )
    else:
        feat_dir = np.array([1, 0, 0])

    fig = go.Figure()

    # Sensitivity surface (colored by sensitivity)
    fig.add_trace(
        go.Scatter3d(
            x=scaled_dirs[:, 0],
            y=scaled_dirs[:, 1],
            z=scaled_dirs[:, 2],
            mode="markers",
            marker=dict(
                size=3,
                color=sensitivities,
                colorscale="Viridis",
                colorbar=dict(title="Sensitivity"),
                opacity=0.7,
            ),
            name="Sensitivity Surface",
        )
    )

    # Feature direction (as a line)
    scale_feat = sensitivities.max() * 6
    fig.add_trace(
        go.Scatter3d(
            x=[0, feat_dir[0] * scale_feat],
            y=[0, feat_dir[1] * scale_feat],
            z=[0, feat_dir[2] * scale_feat],
            mode="lines+markers",
            line=dict(color="red", width=5),
            marker=dict(size=8, color="red"),
            name="Feature Direction (ŵ)",
        )
    )

    # Origin
    fig.add_trace(
        go.Scatter3d(
            x=[0],
            y=[0],
            z=[0],
            mode="markers",
            marker=dict(size=10, color="black"),
            name="Origin",
        )
    )

    fig.update_layout(
        height=600,
        title=f"Sensitivity Rose Plot - Feature {FEAT_TO_VIS_PERTURB} (TiedMLPEncoder)",
        scene=dict(
            xaxis_title="Direction X",
            yaxis_title="Direction Y",
            zaxis_title="Direction Z",
            aspectmode="cube",
        ),
    )
    fig.show()

    # Find max sensitivity direction
    max_idx = np.argmax(sensitivities)
    max_dir = directions_np[max_idx]
    alignment = np.abs(
        np.dot(max_dir, feat_dir) / (np.linalg.norm(max_dir) * np.linalg.norm(feat_dir))
    )
    print(f"Max sensitivity direction: {max_dir}")
    print(f"Feature direction: {feat_dir}")
    print(f"Alignment (|cos θ|): {alignment:.4f}")
else:
    print("No active samples for this feature")

### 4.3 Perturbation Sensitivity Analysis (Medium Scale)

In [ ]:
# Compute perturbation sensitivity for medium scale (sample of features for speed)
N_DIRECTIONS_MED = 500  # More directions needed for 20D space
FEATURES_TO_TEST = list(range(0, N_FEATURES_MED, 10))  # Sample every 10th feature

print("Computing perturbation sensitivity (medium scale, sampled features)...")
print(f"Testing features: {FEATURES_TO_TEST[:10]}... ({len(FEATURES_TO_TEST)} total)")


def compute_sampled_perturbation_sensitivities(
    model, test_samples, feature_indices, n_directions=500, step_size=0.1
):
    """Compute perturbation sensitivity for selected features."""
    results = {}

    with torch.no_grad():
        embeddings = model.ae.encode(test_samples)

    for feat_idx in feature_indices:
        active_mask = test_samples[:, feat_idx] > 0.1
        n_active = active_mask.sum().item()

        if n_active < 20:
            results[feat_idx] = None
            continue

        active_embeddings = embeddings[active_mask][:200]  # Limit samples for speed

        result = perturbation_sensitivity(
            model,
            feat_idx,
            active_embeddings,
            n_directions=n_directions,
            step_size=step_size,
        )
        results[feat_idx] = result

        print(f"Feature {feat_idx}: alignment={result['mean_feature_alignment']:.4f}")

    return results


print("\nTiedLinearRelu:")
relu_perturb_med = compute_sampled_perturbation_sensitivities(
    relu_model_med, test_samples_med, FEATURES_TO_TEST, n_directions=N_DIRECTIONS_MED
)

print("\nTiedMLPEncoder:")
mlp_perturb_med = compute_sampled_perturbation_sensitivities(
    mlp_model_med, test_samples_med, FEATURES_TO_TEST, n_directions=N_DIRECTIONS_MED
)

In [ ]:
# Compare alignment distributions for medium scale
relu_align_med, relu_maxsens_med = extract_alignment_stats(relu_perturb_med)
mlp_align_med, mlp_maxsens_med = extract_alignment_stats(mlp_perturb_med)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Feature Direction Alignment (|cos θ|)",
        "Max Sensitivity Distribution",
    ],
)

fig.add_trace(
    go.Histogram(
        x=relu_align_med,
        nbinsx=50,
        name="TiedLinearRelu",
        marker_color="blue",
        opacity=0.6,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Histogram(
        x=mlp_align_med,
        nbinsx=50,
        name="TiedMLPEncoder",
        marker_color="red",
        opacity=0.6,
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Histogram(
        x=relu_maxsens_med,
        nbinsx=50,
        name="TiedLinearRelu",
        marker_color="blue",
        opacity=0.6,
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Histogram(
        x=mlp_maxsens_med,
        nbinsx=50,
        name="TiedMLPEncoder",
        marker_color="red",
        opacity=0.6,
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.update_xaxes(
    title_text="|cos(θ)| between max-sens dir and feature dir", row=1, col=1
)
fig.update_xaxes(title_text="Max Sensitivity (Δx̂)", row=1, col=2)
fig.update_layout(
    height=400,
    title_text="Perturbation Sensitivity Analysis (Medium Scale)",
    barmode="overlay",
)
fig.show()

print(
    f"TiedLinearRelu: mean alignment={relu_align_med.mean():.4f}, std={relu_align_med.std():.4f}"
)
print(
    f"TiedMLPEncoder: mean alignment={mlp_align_med.mean():.4f}, std={mlp_align_med.std():.4f}"
)

In [ ]:
# Per-feature alignment comparison
def get_per_feature_alignment(perturb_results):
    """Get mean alignment per feature."""
    feat_alignments = {}
    for feat_idx, result in perturb_results.items():
        if result is None:
            continue
        feat_alignments[feat_idx] = result["mean_feature_alignment"]
    return feat_alignments


relu_feat_align = get_per_feature_alignment(relu_perturb_med)
mlp_feat_align = get_per_feature_alignment(mlp_perturb_med)

# Scatter plot: feature alignment vs feature probability
common_features = set(relu_feat_align.keys()) & set(mlp_feat_align.keys())
feat_probs = [ZIPF_PROBS_MED[f] for f in common_features]
relu_aligns = [relu_feat_align[f] for f in common_features]
mlp_aligns = [mlp_feat_align[f] for f in common_features]

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Alignment vs Feature Probability",
        "TiedLinearRelu vs TiedMLPEncoder Alignment",
    ],
)

# Alignment vs probability
fig.add_trace(
    go.Scatter(
        x=feat_probs,
        y=relu_aligns,
        mode="markers",
        name="TiedLinearRelu",
        marker=dict(size=8, color="blue", opacity=0.7),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=feat_probs,
        y=mlp_aligns,
        mode="markers",
        name="TiedMLPEncoder",
        marker=dict(size=8, color="red", opacity=0.7),
    ),
    row=1,
    col=1,
)

# Direct comparison
fig.add_trace(
    go.Scatter(
        x=relu_aligns,
        y=mlp_aligns,
        mode="markers",
        name="Features",
        marker=dict(
            size=8,
            color=feat_probs,
            colorscale="Viridis",
            colorbar=dict(title="p_active"),
            showscale=True,
        ),
        showlegend=False,
    ),
    row=1,
    col=2,
)
# Diagonal line
fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="y=x",
        line=dict(dash="dash", color="gray"),
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.update_xaxes(type="log", title_text="p_active", row=1, col=1)
fig.update_yaxes(title_text="Mean |cos θ| Alignment", row=1, col=1)
fig.update_xaxes(title_text="TiedLinearRelu Alignment", row=1, col=2)
fig.update_yaxes(title_text="TiedMLPEncoder Alignment", row=1, col=2)
fig.update_layout(
    height=400, title_text="Per-Feature Alignment Analysis (Medium Scale)"
)
fig.show()

### 4.4 Test 2 Summary

In [ ]:
print("=" * 80)
print("TEST 2: PERTURBATION SENSITIVITY STRUCTURE - SUMMARY")
print("=" * 80)
print()
print("Feature direction alignment analysis:")
print()
print("Small Scale (n=10, m=3):")
print(
    f"  TiedLinearRelu: mean={relu_align_small.mean():.4f}, std={relu_align_small.std():.4f}"
)
print(
    f"  TiedMLPEncoder: mean={mlp_align_small.mean():.4f}, std={mlp_align_small.std():.4f}"
)
print()
print("Medium Scale (n=200, m=20):")
print(
    f"  TiedLinearRelu: mean={relu_align_med.mean():.4f}, std={relu_align_med.std():.4f}"
)
print(
    f"  TiedMLPEncoder: mean={mlp_align_med.mean():.4f}, std={mlp_align_med.std():.4f}"
)
print()
print("Interpretation:")
print(
    "  - High alignment (~1.0): Sensitive directions align with feature weights -> Direction hypothesis"
)
print(
    "  - Low alignment (~0.0): Sensitive directions are orthogonal to feature weights -> Region hypothesis"
)
print("  - Random alignment (~0.5 for uniform on sphere): No clear structure")
print()

# Interpretation
relu_mean_align = (relu_align_small.mean() + relu_align_med.mean()) / 2
mlp_mean_align = (mlp_align_small.mean() + mlp_align_med.mean()) / 2

print("Conclusion:")
if relu_mean_align > 0.7:
    print(
        f"  TiedLinearRelu: HIGH alignment ({relu_mean_align:.3f}) -> Direction hypothesis supported"
    )
elif relu_mean_align > 0.3:
    print(
        f"  TiedLinearRelu: MODERATE alignment ({relu_mean_align:.3f}) -> Mixed evidence"
    )
else:
    print(
        f"  TiedLinearRelu: LOW alignment ({relu_mean_align:.3f}) -> Region hypothesis supported"
    )

if mlp_mean_align > 0.7:
    print(
        f"  TiedMLPEncoder: HIGH alignment ({mlp_mean_align:.3f}) -> Direction hypothesis supported"
    )
elif mlp_mean_align > 0.3:
    print(
        f"  TiedMLPEncoder: MODERATE alignment ({mlp_mean_align:.3f}) -> Mixed evidence"
    )
else:
    print(
        f"  TiedMLPEncoder: LOW alignment ({mlp_mean_align:.3f}) -> Region hypothesis supported"
    )

## 5. Conclusions

### Motivation

This experiment aimed to distinguish between three hypotheses for how features are encoded in autoencoder embedding spaces:

| Hypothesis | Encoding mechanism | Key prediction |
|---|---|---|
| **Direction** | Linear projection $\hat{w}_i \cdot h$ | Constant gradient norms; sensitivity aligned with weight vectors |
| **Manifold** | Smooth nonlinear function of $h$ | Nonzero gradients everywhere, smoothly varying |
| **Region** | Membership in polytope regions | **Bimodal** gradient norms (near-zero in region interiors, large at boundaries); sensitivity aligned with ReLU boundary normals |

The region hypothesis is motivated by the observation that ReLU networks partition input space into polytopes, each with a unique linear circuit. If features are encoded by region membership rather than direction, this would have significant implications for interpretability methods like SAEs that assume linear feature geometry.

### Results

**Test 1 (Gradient Magnitude Distribution):**
- Neither architecture showed strong bimodality in gradient norms
- Bimodality coefficients remained below the 5/9 threshold for most features
- The "dead gradient fraction" (samples with near-zero gradients) was low across both architectures

**Test 2 (Perturbation Sensitivity):**
- Alignment between maximally-sensitive directions and feature weight vectors showed moderate values
- No clear evidence that sensitive directions align with ReLU boundary normals rather than feature directions

### Interpretation

The results are **inconclusive but lean against the region hypothesis** for these architectures and distributions:

1. **No strong bimodality**: The gradient magnitude distributions do not show the predicted bimodal structure (near-zero in region interiors, large at boundaries). This suggests that even with ReLU nonlinearities, the embedding space doesn't partition into clearly distinct "feature regions."

2. **Metric limitations**: Gradient norm magnitude may not be the most discriminating metric. The piecewise-linear structure of ReLU networks might produce subtler signatures that these tests don't capture.

3. **Architecture constraints**: 
   - `TiedLinearRelu` only applies ReLU to the decoder output, not creating rich polytope structure in the embedding space itself
   - `TiedMLPEncoder` uses LeakyReLU, which attenuates rather than fully zeros gradients, weakening any region-like signatures

### Possible Follow-ups

1. **Input distribution changes**:
   - Use more categorical/binary inputs that might create clearer region boundaries
   - Test with correlated feature distributions (e.g., `CorrelatedPairs`, `DAGDistribution`) where region encoding might be more advantageous

2. **Architecture modifications**:
   - Use hard ReLU throughout (not LeakyReLU) to create genuine dead zones
   - Try deeper networks with more ReLU layers to increase the number of polytope regions
   - Test architectures explicitly designed for region-based encoding

3. **Loss function experiments**:
   - MSE may not incentivize discrete region formation
   - Try quantized reconstruction losses or classification-oriented objectives

4. **Additional tests from the original plan**:
   - **Test 3 (Boundary Normal Clustering)**: Directly examine whether feature activation boundaries are piecewise-linear (clustered normals) vs smooth (continuously rotating normals)
   - **Test 4 (ReLU Boundary Alignment)**: Compute mutual information between neuron activation patterns and feature activations to test if region membership determines features

5. **Scale exploration**:
   - Test at higher compression ratios where superposition pressure might force more creative encoding strategies
   - Vary the sparsity level systematically